In [0]:
%fs ls s3a://my-retail-lakehouse/bronze/

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS retail_lakehouse;

CREATE SCHEMA IF NOT EXISTS retail_lakehouse.bronze;
CREATE SCHEMA IF NOT EXISTS retail_lakehouse.silver;
CREATE SCHEMA IF NOT EXISTS retail_lakehouse.gold;
CREATE SCHEMA IF NOT EXISTS retail_lakehouse.audit;

**Implementing bronze layer which stores data in a raw form without any transformations. Now with the data extracted from external sources we can transform data in next layer.**

In [0]:
%python
# ========================================
# BRONZE LAYER - RAW DATA INGESTION
# Purpose: Extract data from S3 and load into Delta tables
# ========================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

# Base S3 path
base_path = "s3a://my-retail-lakehouse/bronze/"

# Define entities to process
entities = ["customers", "products", "stores", "sales"]

print("=" * 50)
print("STARTING BRONZE LAYER DATA INGESTION")
print("=" * 50)

for entity in entities:
    try:
        print(f"\n[INFO] Processing: {entity}")
        
        # Source path
        source_path = f"{base_path}{entity}/"
        
        # Target table
        target_table = f"retail_lakehouse.bronze.{entity}"
        
        print(f"  - Reading from: {source_path}")
        
        # Read CSV files from S3
        df = spark.read \
            .format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .option("mode", "PERMISSIVE") \
            .option("columnNameOfCorruptRecord", "_rescued_data") \
            .load(source_path)
        
        # Get row count
        row_count = df.count()
        print(f"  - Rows read: {row_count}")
        
        # Write to Delta table (overwrite mode for full refresh)
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)
        
        print(f"  - ✓ Successfully loaded to {target_table}")
        
    except Exception as e:
        print(f"  - ✗ ERROR processing {entity}: {str(e)}")
        raise

print("\n" + "=" * 50)
print("BRONZE LAYER INGESTION COMPLETED")
print("=" * 50)

In [0]:
%python
# ========================================
# BRONZE LAYER VALIDATION
# ========================================

from pyspark.sql.functions import col

print("\n" + "=" * 50)
print("BRONZE LAYER VALIDATION")
print("=" * 50)

entities = ["customers", "products", "stores", "sales"]

for entity in entities:
    table_name = f"retail_lakehouse.bronze.{entity}"
    df = spark.table(table_name)
    count = df.count()
    columns = len(df.columns)
    
    print(f"\n{entity.upper()}:")
    print(f"  - Rows: {count:,}")
    print(f"  - Columns: {columns}")
    
    # Check for rescued data (malformed records)
    if "_rescued_data" in df.columns:
        rescued_count = df.filter(col("_rescued_data").isNotNull()).count()
        if rescued_count > 0:
            print(f"  - ⚠️  Rescued records: {rescued_count}")
        else:
            print(f"  - ✓ No rescued records")

print("\n" + "=" * 50)

**Validate data load**

In [0]:
%python
# Quick row count validation
entities = ["customers", "products", "stores", "sales"]

print("Row Counts:")
print("-" * 30)
for entity in entities:
    count = spark.table(f"retail_lakehouse.bronze.{entity}").count()
    print(f"{entity.capitalize()}: {count:,}")

**Validating data**

In [0]:
%python
# ========================================
# BRONZE LAYER - DETAILED VALIDATION
# ========================================

print("=" * 60)
print("DETAILED BRONZE LAYER VALIDATION WITH SAMPLES")
print("=" * 60)

entities = ["customers", "products", "stores", "sales"]

for entity in entities:
    table_name = f"retail_lakehouse.bronze.{entity}"
    df = spark.table(table_name)
    
    print(f"\n{'=' * 60}")
    print(f"TABLE: {entity.upper()}")
    print(f"{'=' * 60}")
    
    # Row count
    print(f"\nTotal Rows: {df.count():,}")
    
    # Schema
    print("\nSchema:")
    for field in df.schema.fields:
        print(f"  - {field.name}: {field.dataType.simpleString()}")
    
    # Sample data
    print("\nSample Data (First 5 rows):")
    display(df.limit(5))
    
    print("\n" + "-" * 60)

print("\n" + "=" * 60)
print("VALIDATION COMPLETED")
print("=" * 60)